# EDA v7 — 先入観を外した再探索：外部データ・時代背景・残差スキャン**分析日**: 2026年8月16日**位置づけ**: `data_exploration_v6_report.md` に続く第7弾。## 方針v1〜v6 と `submit_result_report.md` 第88・90・95節により、特徴量探索は「もう打ち止め」と記録されている。本ノートブックはその結論を**いったん脇に置き**、以下の3点を新しい観点として検証する。1. **外部データ**（性別・年齢別賃金、賃金トレンド、地域の人口・距離）— 未検証2. **時代背景**（Train 2011-2014 / Test 2014-2017 の暦ドリフト）— 第95節が着手済み、条件付きの大きさは未測定3. **残差スキャン**（現行モデルが取り逃している部分集団の総当たり）— 未実施3 が本ノートブックの中核。従来の探索は「生の効果量が大きい特徴量」を探していたが、それは**既にモデル化済みの信号を再発見しがち**である。現行モデルの OOF 残差に対して部分集団を総当たりすれば、**未捕捉の信号だけが浮く**。## 採否基準（プロジェクト標準、変更しない）- 生の効果量: 定着率差 20%pt 以上 かつ p < 1e-20（`ablation-cannot-settle-feature-blocks`）- 提出ゲート: Test予測の平均絶対差 MAD > 0.02122（ノイズ床95%上限、`refit-chaos-noise-floor`）

In [1]:
import re, sys, warnings, itertools
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')

from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026")
INPUT = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT/"employee_persona_train.csv")
test_persona  = pd.read_csv(INPUT/"employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT/"employee_monthly_train.csv")
test_monthly  = pd.read_csv(INPUT/"employee_monthly_test.csv")
y = train_persona["10年定着ラベル"].values.astype(float)
print(train_persona.shape, test_persona.shape)


(2761, 20) (2502, 19)


In [2]:
# 54_ のブロック L2 / M を最小構成で再現（本ノートブックを自己完結させるため）
def extract_workstyle_section(text):
    if pd.isna(text): return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m: return m.group(1).strip()
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None

NEG = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")
def classify_reloc(s):
    if s is None: return None
    if NEG.search(s): return False
    if POS.search(s): return True
    return None

def desired_v1(s):
    if s is None: return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m: return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None

def desired_v2(s):
    if s is None: return None
    loc = desired_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    return loc.strip("「」") if loc is not None else None

_AM = {"情報", "理工学"}
_AJ = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}

def l2m_block(p):
    ws = p["入社時メモ"].apply(extract_workstyle_section)
    reloc = ws.apply(classify_reloc); desired = ws.apply(desired_v2)
    match = (desired == p["初期勤務地"]) & desired.notna()
    rt, rf = reloc == True, reloc == False
    valid = desired.notna() & reloc.notna()
    state = pd.Series("unknown", index=p.index)
    state[valid & rt &  match] = "許容_一致";   state[valid & rt & ~match] = "許容_不一致"
    state[valid & rf &  match] = "非許容_一致"; state[valid & rf & ~match] = "非許容_不一致"
    l2 = (state == "非許容_不一致").astype(int)
    mb = ((~p["専攻分野"].isin(_AM)) & p["初期職種"].isin(_AJ)).astype(int)
    return pd.DataFrame({"社員ID": p["社員ID"].values, "転居x勤務地_状態_v2": state.values,
                         "M_不適合": mb.values, "L2xM_リスク要因数": (l2+mb).values,
                         "希望勤務地": desired.values, "転居許容": reloc.values})

AGG_METRICS = ["残業時間","有給取得日数","欠勤日数","研修時間","上司との面談実施回数","情報共有件数",
  "在宅勤務日数","360度評価_親和度","360度評価_信頼度","360度評価_主体度","360度評価_学習度",
  "360度評価_共有貢献度","360度評価者数","顧客満足度評価","担当プロジェクト数","月例給与_円"]

def monthly_agg(m):
    a = m.groupby("社員ID")[AGG_METRICS].agg(['mean','std','min','max','median','last'])
    a.columns = ['_'.join(c) for c in a.columns]
    a['n_months'] = m.groupby("社員ID").size()
    a['在籍_休職月数'] = m.assign(x=(m['月末在籍状態']=='休職').astype(int)).groupby('社員ID')['x'].sum()
    return a.reset_index()

def build(p, m):
    p = p.copy(); p["入社日"] = pd.to_datetime(p["入社日"])
    p["入社年"] = p["入社日"].dt.year; p["入社月"] = p["入社日"].dt.month
    p["fy"] = p["入社年"] - (p["入社月"] < 4)          # 年度（4月始まり）
    for c in ["入社時メモ","上司からのフィードバック","同僚からのフィードバック"]:
        p[c+"_len"] = p[c].fillna("").astype(str).str.len()
    return p.merge(l2m_block(p), on="社員ID").merge(monthly_agg(m), on="社員ID", how="left")

TR = build(train_persona, train_monthly)
TE = build(test_persona,  test_monthly)
print(TR.shape, TE.shape)


(2761, 129) (2502, 128)


---# 1. 共変量シフトの全数監査 — 動いているのは給与「だけ」第95節は adversarial validation の AUC=0.824 と `初任給` の重要度1位を報告したが、**どの列がどれだけ動いているかの全数監査**はしていなかった。全数値列についてTrain/Test 間の Cohen's d を測る。

In [3]:
num = ["年齢","月例給与_円","360度評価_親和度","360度評価_信頼度","360度評価_主体度","360度評価_学習度",
       "360度評価_共有貢献度","360度評価更新フラグ","360度評価者数","担当プロジェクト数","顧客満足度評価",
       "残業時間","有給取得日数","欠勤日数","研修時間","上司との面談実施回数","情報共有件数","在宅勤務日数"]
A = train_monthly.groupby("社員ID")[num].agg(['mean','std','max']); A.columns=['_'.join(c) for c in A.columns]
B = test_monthly.groupby("社員ID")[num].agg(['mean','std','max']);  B.columns=['_'.join(c) for c in B.columns]
rows=[]
for c in A.columns:
    a, b = A[c].dropna(), B[c].dropna()
    if len(a)<50 or len(b)<50: continue
    sp = np.sqrt((a.var()+b.var())/2)
    rows.append((c, a.mean(), b.mean(), (b.mean()-a.mean())/sp if sp>0 else 0))
for c in ["入社時年齢","前職経験月数","初任給_円"]:
    a, b = train_persona[c].astype(float), test_persona[c].astype(float)
    rows.append((c+"(persona)", a.mean(), b.mean(),
                 (b.mean()-a.mean())/np.sqrt((a.var()+b.var())/2)))
shift = pd.DataFrame(rows, columns=["col","train_mean","test_mean","cohen_d"]).sort_values("cohen_d", key=abs, ascending=False)
print(shift.head(12).to_string(index=False))
print("\n|d|>0.2: %d列 / %d列   |d|>0.5: %d列" % ((shift.cohen_d.abs()>0.2).sum(), len(shift), (shift.cohen_d.abs()>0.5).sum()))
print("給与系を除いて |d|>0.15 の列:", shift[(shift.cohen_d.abs()>0.15)&(~shift.col.str.contains('給与|初任給'))].col.tolist())


             col    train_mean     test_mean   cohen_d
      月例給与_円_std   3529.595590   4104.115517  0.307328
  初任給_円(persona) 248226.729446 262812.949640  0.252761
      月例給与_円_max 289377.761681 305626.298961  0.228117
     月例給与_円_mean 285227.918426 300757.094325  0.225600
360度評価更新フラグ_mean      0.162111      0.163336  0.100251
 360度評価更新フラグ_std      0.376095      0.377252  0.091658
       残業時間_mean     16.183537     15.459632 -0.073965
          年齢_std      0.651927      0.656258  0.065836
     有給取得日数_mean      0.839895      0.846739  0.058391
          年齢_max     26.165520     26.406475  0.057091
         年齢_mean     25.206520     25.440814  0.055496
  入社時年齢(persona)     24.265121     24.486811  0.052564

|d|>0.2: 4列 / 57列   |d|>0.5: 0列
給与系を除いて |d|>0.15 の列: []


**結果**: `|d|>0.2` は給与4列のみ。給与以外で `|d|>0.15` の列は**ゼロ**。このデータセットの共変量シフトは給与に完全に局在している。

In [4]:
# 周辺分布ではなく「条件付き」で測ると、シフトははるかに大きい
print("=== 新卒 × 学歴 別の初任給シフト（条件付き）===")
for src, d in [("train", train_persona), ("test", test_persona)]:
    d["fy"] = pd.to_datetime(d["入社日"]).dt.year - (pd.to_datetime(d["入社日"]).dt.month < 4)
sh = pd.concat([train_persona.assign(src="train"), test_persona.assign(src="test")], ignore_index=True)
n = sh[sh["入社区分"]=="新卒"]
for ed, g in n.groupby("最終学歴"):
    a = g[g.src=="train"]["初任給_円"]; b = g[g.src=="test"]["初任給_円"]
    print(f"  {ed:8s} train {a.mean():8.0f}(sd {a.std():6.0f}) → test {b.mean():8.0f}  シフト {(b.mean()-a.mean()):+7.0f}円 = {(b.mean()-a.mean())/a.std():+.2f} sd")
print("\n=== 中途 × 等級 ===")
m = sh[sh["入社区分"]=="中途"]
for gr, g in m.groupby("初期等級"):
    a = g[g.src=="train"]["初任給_円"]; b = g[g.src=="test"]["初任給_円"]
    if len(a)<20 or len(b)<20: continue
    print(f"  {gr:4s} train {a.mean():8.0f}(sd {a.std():6.0f}) → test {b.mean():8.0f}  シフト {(b.mean()-a.mean()):+7.0f}円 = {(b.mean()-a.mean())/a.std():+.2f} sd")


=== 新卒 × 学歴 別の初任給シフト（条件付き）===
  大学卒      train   224188(sd  11244) → test   235652  シフト  +11465円 = +1.02 sd
  大学院卒     train   255479(sd  12878) → test   267493  シフト  +12014円 = +0.93 sd
  専門学校卒    train   198305(sd   9547) → test   208098  シフト   +9793円 = +1.03 sd
  短大・高専卒   train   201484(sd  10342) → test   210792  シフト   +9308円 = +0.90 sd
  高校卒      train   176010(sd   8853) → test   186140  シフト  +10131円 = +1.14 sd

=== 中途 × 等級 ===
  G1   train   238219(sd  18829) → test   250839  シフト  +12620円 = +0.67 sd
  G2   train   276960(sd  24817) → test   289954  シフト  +12994円 = +0.52 sd
  G3   train   336064(sd  26194) → test   351498  シフト  +15434円 = +0.59 sd
  G4   train   403843(sd  34512) → test   427162  シフト  +23319円 = +0.68 sd


**新発見（第95節の精緻化）**: 第95節が報告した `+14,586円` は**周辺分布**の差である。学歴・等級で条件付けると **新卒は +0.90〜1.14 sd、中途は +0.52〜0.68 sd** とはるかに大きい。周辺分布では学歴間の分散（大きい）に薄められて `d=0.25` にしか見えないため、**周辺分布の監査だけではこの大きさを見落とす**。木は絶対値の閾値で分割するので、たとえば「新卒大学卒で 初任給 > 23万円」という分岐はTrain では上位3割を切るが、Test では**上位7割**を切ってしまう。

---# 2. 給与の効果は「絶対額」ではなく「コホート内の相対位置」シフトが実害を持つのは、給与が定着と関係している場合に限る。まずその形を確かめる。

In [5]:
from sklearn.metrics import roc_auc_score
T = TR.copy()
nsotsu = T[T["入社区分"]=="新卒"].copy()
nsotsu["z_ed"]    = nsotsu.groupby(["最終学歴"])["初任給_円"].transform(lambda x:(x-x.mean())/x.std())
nsotsu["z_fy_ed"] = nsotsu.groupby(["fy","最終学歴"])["初任給_円"].transform(lambda x:(x-x.mean())/x.std())
yy = y[nsotsu.index]
print("=== 新卒（n=%d）: 給与表現ごとの単変量AUC ===" % len(nsotsu))
for c in ["初任給_円","z_ed","z_fy_ed"]:
    print(f"  {c:10s} AUC={roc_auc_score(yy, -nsotsu[c]):.4f}  corr={np.corrcoef(nsotsu[c],yy)[0,1]:+.4f}")

nsotsu["q"] = pd.qcut(nsotsu["z_fy_ed"], 5, labels=False)
g = nsotsu.assign(y=yy).groupby("q")["y"].agg(["mean","count"]); print("\n", g)
ct = pd.crosstab(nsotsu["q"], yy); obs = ct.values
sc = np.arange(5); N=obs.sum(); R=obs[:,1].sum(); ni=obs.sum(1); pb=R/N
z = (sc*(obs[:,1]-ni*pb)).sum()/np.sqrt(pb*(1-pb)*((ni*sc**2).sum()-((ni*sc).sum())**2/N))
print("上下差 %.1f pt / Cochran-Armitage z=%.2f p=%.3g" % ((g['mean'].iloc[0]-g['mean'].iloc[-1])*100, z, 2*stats.norm.sf(abs(z))))


=== 新卒（n=2025）: 給与表現ごとの単変量AUC ===
  初任給_円      AUC=0.5320  corr=-0.0436
  z_ed       AUC=0.5923  corr=-0.1585
  z_fy_ed    AUC=0.5962  corr=-0.1649

        mean  count
q                 
0  0.632353    408
1  0.539801    402
2  0.559113    406
3  0.483951    405
4  0.386139    404
上下差 24.6 pt / Cochran-Armitage z=-6.99 p=2.73e-12


In [6]:
# 絶対額を固定したまま入社年度を動かすと定着率が変わるか（=相対説の直接検定）
s = T[(T["入社区分"]=="新卒")&(T["最終学歴"]=="大学卒")].copy(); s["y"]=y[s.index]
s["bin"] = pd.cut(s["初任給_円"], [0,210000,218000,224000,230000,238000,999999])
print("=== 新卒大学卒: 絶対額ビン × 入社年度 の定着率 ===")
print(s.pivot_table(index="bin", columns="fy", values="y", aggfunc=["mean","count"]).round(3))


=== 新卒大学卒: 絶対額ビン × 入社年度 の定着率 ===


                   mean               count          
fy                 2011   2012   2013  2011 2012 2013
bin                                                  
(0, 210000]       0.754  0.672  0.644    57   58   45
(210000, 218000]  0.551  0.628  0.558    98   86   52
(218000, 224000]  0.550  0.485  0.574    80   68   54
(224000, 230000]  0.462  0.553  0.522    65   85   67
(230000, 238000]  0.272  0.407  0.484    81  108  126
(238000, 999999]  0.200  0.308  0.452    15   39   42


**結果**: 新卒では `初任給` の絶対額は AUC 0.532 とほぼ無力だが、**コホート内 z にすると 0.596** まで上がる。効果量は五分位の上下で **24.6pt**（Cochran-Armitage p=2.7e-12）。さらに絶対額ビンを固定して年度を動かすと、高額ビン（23.0-23.8万）の定着率が`2011: 0.272 → 2012: 0.407 → 2013: 0.484` と上がる。**同じ23万円でも、年が進むほど「相対的に高給ではなくなる」ため定着が下がらない** ＝ 効くのは相対位置である。ここまでは「シフトを補正すべき」という筋書きに完全に整合する。次節でその補正を実測する。

---# 3. しかし補正しても効かない（CatBoost実測） — 第95節の追認コホート正規化した給与列を作り、単層CatBoost（`54_` と同じ A_PARAMS・560反復・5シード）でTest予測がどれだけ動くかを測る。**判定はラベルを見ない指標**（MAD vs ノイズ床 0.02122）で行う（`validation-asymmetry`: 検証の「改善」は信用しない）。> このセルはローカルで約1分/構成。Colab不要。

In [7]:
import catboost as cb
from sklearn.metrics import log_loss

ALL = pd.concat([TR.drop(columns=["10年定着ラベル"]), TE], ignore_index=True); nTR = len(TR)
ALL["cohort"] = ALL["入社区分"] + "|" + np.where(ALL["入社区分"]=="新卒", ALL["最終学歴"], ALL["初期等級"].astype(str))
g = ALL.groupby(["cohort","fy"])["初任給_円"]                      # ← ラベル不使用（train+testの特徴量のみ）
ALL["初任給_コホート年度内z"] = ((ALL["初任給_円"]-g.transform("mean"))/g.transform("std")).values
ALL["初任給_コホート年度比"]   = (ALL["初任給_円"]/g.transform("mean")).values
idx = ALL.groupby("fy")["初任給_円"].transform("mean"); base = ALL.loc[ALL.fy==2011,"初任給_円"].mean()
ALL["賃金指数"] = (idx/base).values
for c in ["月例給与_円_mean","月例給与_円_max","月例給与_円_last"]:
    ALL[c+"_実質"] = ALL[c]/ALL["賃金指数"]
ALL["初任給_実質"] = ALL["初任給_円"]/ALL["賃金指数"]

TRf, TEf = ALL[:nTR].copy(), ALL[nTR:].copy()
DEFL = ["初任給_コホート年度内z","初任給_コホート年度比","初任給_実質",
        "月例給与_円_mean_実質","月例給与_円_max_実質","月例給与_円_last_実質","賃金指数"]
DROP = ["社員ID","入社日","入社時メモ","上司からのフィードバック","同僚からのフィードバック","希望勤務地","cohort"]
CONFIGS = {
  "C0_baseline": {"add": [], "drop": []},
  "C1_add_z":    {"add": ["初任給_コホート年度内z"], "drop": []},
  "C2_add_all":  {"add": DEFL, "drop": []},
  "C3_replace":  {"add": ["初任給_コホート年度内z","月例給与_円_mean_実質","月例給与_円_max_実質","月例給与_円_last_実質"],
                  "drop": ["初任給_円","月例給与_円_mean","月例給与_円_max","月例給与_円_last"]},
}
A_PARAMS = dict(depth=4, learning_rate=0.03518359458951149, l2_leaf_reg=2.217690447016724,
                border_count=218, bagging_temperature=0.6787467566574921, random_strength=1.438494697238285)
SEEDS=[42,2024,7,1234,99]; NITER=560
early = set(train_monthly.loc[train_monthly["月末在籍状態"]=="退職","社員ID"].unique())
order = np.argsort(TRf["入社日"].values, kind="stable"); cut=int(len(TRf)*0.8)
tr_idx, va_idx = order[:cut], order[cut:]
surv = ~TRf["社員ID"].isin(early).values
va_idx = np.array([i for i in va_idx if surv[i]])        # 生存者のみで検証（40_プロトコル）

results={}
for name,cfg in CONFIGS.items():
    cols=[c for c in TRf.columns if c not in DROP+["10年定着ラベル","fy"]]
    cols=[c for c in cols if (c not in DEFL or c in cfg["add"]) and c not in cfg["drop"]]
    Xtr, Xte = TRf[cols].copy(), TEf[cols].copy()
    obj=[c for c in cols if Xtr[c].dtype==object or Xtr[c].dtype==bool]
    for c in obj: Xtr[c]=Xtr[c].astype(str); Xte[c]=Xte[c].astype(str)
    numc=[c for c in cols if c not in obj]; Xtr[numc]=Xtr[numc].fillna(-999); Xte[numc]=Xte[numc].fillna(-999)
    vp,tp=[],[]
    for s in SEEDS:
        m=cb.CatBoostClassifier(**A_PARAMS,iterations=NITER,random_seed=s,verbose=False,cat_features=obj,task_type="CPU")
        m.fit(Xtr.iloc[tr_idx], y[tr_idx]); vp.append(m.predict_proba(Xtr.iloc[va_idx])[:,1])
        m2=cb.CatBoostClassifier(**A_PARAMS,iterations=int(NITER*1.25),random_seed=s,verbose=False,cat_features=obj,task_type="CPU")
        m2.fit(Xtr, y); tp.append(m2.predict_proba(Xte)[:,1])
    results[name]=dict(val=log_loss(y[va_idx], np.mean(vp,0)), test=np.mean(tp,0), ncol=len(cols))
    print(f"{name:14s} ncol={len(cols):3d}  val={results[name]['val']:.6f}")

print("\n=== 対 C0 の Test予測 平均絶対差（ノイズ床95%上限 0.02122）===")
b=results["C0_baseline"]["test"]
for k,v in results.items():
    if k=="C0_baseline": continue
    print(f"  {k:14s} MAD={np.abs(v['test']-b).mean():.5f}  corr={np.corrcoef(v['test'],b)[0,1]:.5f}  Δval={v['val']-results['C0_baseline']['val']:+.6f}")


C0_baseline    ncol=121  val=0.507833


C1_add_z       ncol=122  val=0.509186


C2_add_all     ncol=128  val=0.505735


C3_replace     ncol=121  val=0.508609

=== 対 C0 の Test予測 平均絶対差（ノイズ床95%上限 0.02122）===
  C1_add_z       MAD=0.01452  corr=0.99749  Δval=+0.001353
  C2_add_all     MAD=0.01557  corr=0.99718  Δval=-0.002098
  C3_replace     MAD=0.01607  corr=0.99698  Δval=+0.000777


**結果（不採用）**: 3構成とも **MAD 0.0145〜0.0161 < ノイズ床95%上限 0.02122**。提出ゲートを通らない ＝ 再学習の揺らぎと区別できず、提出しても現最良の近傍に戻るだけ。（検証スコアの向きは構成によってバラバラだが、`validation-asymmetry` の通りそこは判断材料にしない。）第95節の結論「シフトは実在するが P(y|x) は安定しており補正の余地が無い」を、**別の実装・別の判定指標（ラベル非依存のMAD）で独立に追認**した。条件付きシフトが 1 sd あってもモデルが動かないのは、CatBoost（depth=4・強正則化）が`最終学歴`・`初期等級` を主に使い、給与の限界寄与が小さいため。

---# 4. 外部データの検証 — ユーザー提案の「性別・年齢別の平均年収」ほか外部データを持ち込む前に、**そのデータが表現する構造が本データに存在するか**を確かめる。本データは合成であり（第88節で生成規則を逆解析済み）、生成器が実世界の賃金構造を使っていなければ、実統計のマージは**存在しない差を注入するだけ**になる。

In [8]:
print("########## 4-1. 性別による賃金差は存在するか ##########")
for k,g in train_persona.groupby(["入社区分","最終学歴"]):
    if len(g)<80: continue
    m=g[g.性別=="男性"]["初任給_円"].mean(); f=g[g.性別=="女性"]["初任給_円"].mean()
    print(f"  {k[0]}/{k[1]:8s} 男{m:8.0f} 女{f:8.0f}  女/男={f/m:.4f}")
print("  （実日本の同年齢・同学歴の女/男比は概ね 0.85〜0.90）")

print("\n########## 4-2. 性別による行動差は存在するか ##########")
agg = train_monthly.groupby("社員ID")[["残業時間","在宅勤務日数","有給取得日数","欠勤日数"]].mean()
gr  = train_monthly.groupby("社員ID")["月例給与_円"].agg(["first","last"]); gr["昇給率"]=gr["last"]/gr["first"]
j = train_persona.set_index("社員ID").join(agg).join(gr[["昇給率"]])
print(j.groupby("性別")[["残業時間","在宅勤務日数","有給取得日数","欠勤日数","昇給率"]].mean().round(4))

print("\n########## 4-3. 性別 × 定着率 ##########")
print(pd.crosstab(train_persona["性別"], y, normalize="index").round(4))
print("chi2 p=%.4f  (差 %.1f pt — 採用基準 20pt には遠い。かつ 性別 は既に生カテゴリとして投入済み)"
      % (stats.chi2_contingency(pd.crosstab(train_persona["性別"], y))[1],
         abs(np.diff(pd.crosstab(train_persona["性別"], y, normalize="index")[1.0].values))[0]*100))


########## 4-1. 性別による賃金差は存在するか ##########
  中途/大学卒      男  315060 女  315145  女/男=1.0003
  中途/大学院卒     男  316767 女  305535  女/男=0.9645
  新卒/大学卒      男  223319 女  225259  女/男=1.0087
  新卒/大学院卒     男  254855 女  256213  女/男=1.0053
  新卒/専門学校卒    男  197250 女  199571  女/男=1.0118
  新卒/短大・高専卒   男  200753 女  202299  女/男=1.0077
  新卒/高校卒      男  174267 女  178386  女/男=1.0236
  （実日本の同年齢・同学歴の女/男比は概ね 0.85〜0.90）

########## 4-2. 性別による行動差は存在するか ##########
       残業時間  在宅勤務日数  有給取得日数    欠勤日数     昇給率
性別                                         
女性  16.3939  5.4531  0.8387  0.0682  1.0269
男性  16.0123  5.3169  0.8408  0.0682  1.0273

########## 4-3. 性別 × 定着率 ##########
col_0     0.0     1.0
性別                   
女性     0.4609  0.5391
男性     0.4146  0.5854
chi2 p=0.0164  (差 4.6 pt — 採用基準 20pt には遠い。かつ 性別 は既に生カテゴリとして投入済み)


In [9]:
print("########## 4-4. 賃金トレンド: 本データ vs 実日本 ##########")
real = {2011:202.0, 2012:201.8, 2013:198.0, 2014:200.4, 2015:202.0, 2016:203.4}  # 厚労省 賃金構造基本統計調査 大卒初任給(千円)
sh2 = pd.concat([train_persona, test_persona], ignore_index=True)
sh2["fy"] = pd.to_datetime(sh2["入社日"]).dt.year - (pd.to_datetime(sh2["入社日"]).dt.month < 4)
syn = sh2[(sh2["入社区分"]=="新卒")&(sh2["最終学歴"]=="大学卒")].groupby("fy")["初任給_円"].mean()
d = pd.DataFrame({"本データ": syn.round(0), "実日本": pd.Series(real)*1000})
d["本データ_2011比"]=(d["本データ"]/d["本データ"].iloc[0]).round(4)
d["実日本_2011比"]  =(d["実日本"]/d["実日本"].iloc[0]).round(4)
print(d.to_string())
print("\n年率: 本データ %.2f%%/年  vs  実日本 %.2f%%/年  （約10倍の乖離）"
      % (((d["本データ"].iloc[-1]/d["本データ"].iloc[0])**(1/5)-1)*100,
         ((d["実日本"].iloc[-1]/d["実日本"].iloc[0])**(1/5)-1)*100))


########## 4-4. 賃金トレンド: 本データ vs 実日本 ##########
          本データ       実日本  本データ_2011比  実日本_2011比
2011  222048.0  202000.0      1.0000     1.0000
2012  224203.0  201800.0      1.0097     0.9990
2013  226365.0  198000.0      1.0194     0.9802
2014  232685.0  200400.0      1.0479     0.9921
2015  235251.0  202000.0      1.0595     1.0000
2016  238781.0  203400.0      1.0754     1.0069

年率: 本データ 1.46%/年  vs  実日本 0.14%/年  （約10倍の乖離）


**判定: 外部の賃金統計は使えない（不採用）**| 検証 | 本データ | 実日本 | 判定 ||---|---|---|---|| 女/男 の初任給比 | **0.96〜1.02** | 0.85〜0.90 | 性別賃金差は**存在しない** || 性別 × 残業/在宅/有給/欠勤/昇給率 | ほぼ完全に同一 | 差あり | 性別行動差も**存在しない** || 大卒初任給の年上昇率 | **1.46%/年** | 0.14%/年 | **約10倍**の乖離 |生成器は実世界の賃金構造（性別間格差・実際のインフレ率）を一切使っていない。したがって「性別・年齢別の平均年収」をマージすると、**本データに存在しない 10〜15% の男女差を特徴量に注入する**ことになり、有害である。暦補正が必要な場合も、実日本の指数では 1/10 しか補正できず、第3節の**内部コホート基準が厳密に上位互換**。なお 性別 × 定着率は 4.6pt (p=0.016) の差があるが、採用基準 20pt に遠く、かつ `性別` は既に生カテゴリとしてモデルに入っている。

---# 4-5. テキスト3列は「テンプレート」か「自由文」かv6 は n-gram マイニングで decisive な null を出したが、**合成データなら少数の文テンプレートから生成されている可能性**があり、その場合はテンプレートIDを categorical にする方が n-gram よりはるかに強い。テンプレート集合が実在するかを確かめる。

In [10]:
print("=== 列そのもののユニーク率 ===")
for c in ["入社時メモ","上司からのフィードバック","同僚からのフィードバック"]:
    a=train_persona[c].fillna(""); print(f"  {c}: n={len(a)} unique={a.nunique()} ({a.nunique()/len(a):.1%})")

print("\n=== 文単位に分解したときのユニーク文数 ===")
def sents(s): return [x.strip() for x in re.split(r"[。\n]", str(s)) if x.strip()]
for c in ["上司からのフィードバック","同僚からのフィードバック"]:
    allsent=[]
    for t in pd.concat([train_persona[c],test_persona[c]]).fillna(""): allsent+=sents(t)
    vc=pd.Series(allsent).value_counts()
    print(f"  {c}: 総文数 {len(allsent)} / ユニーク {len(vc)} ({len(vc)/len(allsent):.2%}) / 最頻文の出現回数 {vc.iloc[0]}")

print("\n=== 入社時メモ: セクション構成 × 定着率 ===")
secs=train_persona["入社時メモ"].apply(lambda t: tuple(sorted(set(re.findall(r"・([^：]{2,8})：", str(t))))))
d=pd.DataFrame({"p":secs.astype(str),"y":y})
print(d.groupby("p")["y"].agg(["mean","count"]).query("count>=30").sort_values("mean").to_string())
print("chi2 p=%.3f" % stats.chi2_contingency(pd.crosstab(secs.astype(str), y))[1])


=== 列そのもののユニーク率 ===
  入社時メモ: n=2761 unique=2761 (100.0%)
  上司からのフィードバック: n=2761 unique=2761 (100.0%)
  同僚からのフィードバック: n=2761 unique=2761 (100.0%)

=== 文単位に分解したときのユニーク文数 ===
  上司からのフィードバック: 総文数 35599 / ユニーク 35591 (99.98%) / 最頻文の出現回数 2
  同僚からのフィードバック: 総文数 19616 / ユニーク 19528 (99.55%) / 最頻文の出現回数 8

=== 入社時メモ: セクション構成 × 定着率 ===
                                         mean  count
p                                                   
('キャリア志向', '勤務地・働き方', '経歴')          0.559322     59
('キャリア志向', '人物所見', '勤務地・働き方', '経歴')  0.563264   2537
('人物所見',)                            0.575949    158
chi2 p=0.536


**結果（不採用）**: 上司フィードバックは **35,599文中35,591文（99.98%）がユニーク**、同僚フィードバックも 99.5% がユニーク。最頻の文でも出現2〜8回にとどまる。**テンプレート集合は存在せず、LLM生成の自由文**である。セクション構成の違いも p=0.54 で無信号。テキスト系は n-gram（v6）だけでなく**テンプレート分解の観点からも閉じている**。

---# 5. 残差スキャン — 現行モデルが取り逃している部分集団の総当たりここが本ノートブックの中核。現行モデルの OOF 予測を作り、**残差 `y - p` が系統的に0 からずれる部分集団**を総当たりで探す。生の効果量スキャン（既にモデル化済みの信号を再発見しがち）と違い、**未捕捉の信号だけが浮く**。地域の外部データ（都道府県人口・東京からの距離・希望地との人口比）も候補に含める。

In [11]:
from sklearn.model_selection import StratifiedKFold
DROPC=["社員ID","入社日","入社時メモ","上司からのフィードバック","同僚からのフィードバック","希望勤務地","10年定着ラベル","fy"]
cols=[c for c in TR.columns if c not in DROPC]
X=TR[cols].copy(); obj=[c for c in cols if X[c].dtype==object or X[c].dtype==bool]
for c in obj: X[c]=X[c].astype(str)
X[[c for c in cols if c not in obj]] = X[[c for c in cols if c not in obj]].fillna(-999)

oof=np.zeros(len(X))
for tri,vai in StratifiedKFold(5,shuffle=True,random_state=0).split(X,y):
    ps=[]
    for s in (42,7,2024):
        m=cb.CatBoostClassifier(**A_PARAMS,iterations=560,random_seed=s,verbose=False,cat_features=obj,task_type="CPU")
        m.fit(X.iloc[tri],y[tri]); ps.append(m.predict_proba(X.iloc[vai])[:,1])
    oof[vai]=np.mean(ps,0)
res = y - oof
print("OOF logloss = %.5f （定数ベースライン %.5f）" % (log_loss(y,oof), log_loss(y,np.full(len(y),y.mean()))))


OOF logloss = 0.49194 （定数ベースライン 0.68476）


In [12]:
# 外部データ（地域）: 都道府県人口(千人, 2015国勢調査) と 東京からの直線距離(km)
POP  = {"東京":13515,"大阪":8839,"愛知":7483,"福岡":5102,"北海道":5382,"仙台":2334,"その他":np.nan}
DIST = {"東京":0,"大阪":400,"愛知":270,"福岡":880,"北海道":830,"仙台":300,"その他":np.nan}
P = TR.copy(); P["res"]=res
P["配属地_人口"]=P["初期勤務地"].map(POP); P["配属地_東京距離"]=P["初期勤務地"].map(DIST)
P["希望地_人口"]=P["希望勤務地"].map(POP); P["人口比_配属÷希望"]=P["配属地_人口"]/P["希望地_人口"]
P["cohort"]=P["入社区分"]+"|"+np.where(P["入社区分"]=="新卒",P["最終学歴"],P["初期等級"].astype(str))
gg=P.groupby(["cohort","fy"])["初任給_円"]; P["初任給_z"]=(P["初任給_円"]-gg.transform("mean"))/gg.transform("std")

cands={}
for c in ["入社区分","最終学歴","専攻分野","前職職種","採用経路","性別","初期職種","初期勤務地",
          "初期等級","初期役割","転居x勤務地_状態_v2","希望勤務地","fy","入社月"]:
    for v in P[c].dropna().unique():
        m=(P[c]==v).values
        if 100<=m.sum()<=len(P)-100: cands[f"{c}=={v}"]=m
for c,thr in [("初任給_z",[-1,-.5,0,.5,1]),("入社時年齢",[22,24,26,30]),("前職経験月数",[1,24,60,120]),
              ("配属地_人口",[3000,6000]),("配属地_東京距離",[100,500]),("人口比_配属÷希望",[0.5,1.0,2.0]),
              ("残業時間_mean",[10,15,20,25]),("月例給与_円_mean",[250000,300000,350000]),
              ("在宅勤務日数_mean",[2,5,8]),("360度評価_信頼度_mean",[3.2,3.5,3.8])]:
    for t in thr:
        m=(P[c]>t).fillna(False).values
        if 100<=m.sum()<=len(P)-100: cands[f"{c}>{t}"]=m

rows=[(k,m.sum(),res[m].mean()-res[~m].mean(),stats.ttest_ind(res[m],res[~m],equal_var=False)[1])
      for k,m in cands.items()]
R=pd.DataFrame(rows,columns=["subgroup","n","res_diff","p"]).sort_values("p")
print("単変量: 候補 %d 件 / Bonferroni閾値 p<%.2g" % (len(cands), 0.05/len(cands)))
print(R.head(10).to_string(index=False, float_format=lambda x:f"{x:.4g}"))
print("\n>>> Bonferroni を通った件数: %d" % (R.p < 0.05/len(cands)).sum())


単変量: 候補 94 件 / Bonferroni閾値 p<0.00053
               subgroup    n  res_diff       p
               専攻分野==情報  422  -0.05275 0.01404
初期職種==データ・商品企画・コンサルティング  268   0.04899 0.05873
            採用経路==インターン  298  -0.04151 0.09916
           配属地_東京距離>500  350  -0.03818  0.1092
             初期勤務地==北海道  118  -0.06078  0.1142
              初期勤務地==大阪  394   0.03363  0.1232
            配属地_人口>6000 2046   0.02687  0.1252
              初期勤務地==仙台  124  -0.05896  0.1274
             最終学歴==大学院卒  460   0.03057  0.1408
             希望勤務地==その他  192   0.03954  0.1463

>>> Bonferroni を通った件数: 0


In [13]:
# 2変数交互作用
names=[k for k,m in cands.items() if m.sum()>=150]
rows=[]
for a,b in itertools.combinations(names,2):
    m=cands[a]&cands[b]
    if m.sum()<80 or m.sum()>len(P)-80: continue
    rows.append((f"{a} & {b}", m.sum(), res[m].mean()-res[~m].mean(),
                 stats.ttest_ind(res[m],res[~m],equal_var=False)[1]))
R2=pd.DataFrame(rows,columns=["subgroup","n","res_diff","p"]).sort_values("p")
thr=0.05/len(R2)
print("交互作用: 検定 %d 件 / Bonferroni閾値 p<%.2g" % (len(R2), thr))
print(R2.head(10).to_string(index=False, float_format=lambda x:f"{x:.4g}"))
print("\n>>> Bonferroni を通った件数: %d" % (R2.p<thr).sum())


交互作用: 検定 2110 件 / Bonferroni閾値 p<2.4e-05
                           subgroup   n  res_diff         p
              専攻分野==その他 & 初任給_z>0.5  90    0.1592  0.000361
            fy==2012 & 残業時間_mean>25  99  -0.07592 0.0009234
 初期職種==データ・商品企画・コンサルティング & fy==2013  89    0.1301  0.001556
                専攻分野==情報 & 初任給_z>-1 360  -0.06645  0.003712
    初期勤務地==大阪 & 転居x勤務地_状態_v2==許容_一致 115   0.09473  0.006283
    転居x勤務地_状態_v2==許容_一致 & 希望勤務地==大阪 115   0.09473  0.006283
          専攻分野==その他 & 在宅勤務日数_mean>5 100    0.1228  0.006832
初期職種==データ・商品企画・コンサルティング & 初任給_z>0.5 202   0.07736  0.008026
     性別==女性 & 転居x勤務地_状態_v2==非許容_不一致 176  -0.07098  0.008958
          希望勤務地==大阪 & 在宅勤務日数_mean>5  93   0.08402  0.009268

>>> Bonferroni を通った件数: 0


**結果（決定的な null）**| スキャン | 検定数 | Bonferroni閾値 | 最小p | **通過数** ||---|---|---|---|---|| 単変量 | 94 | 5.3e-4 | 0.014 | **0** || 2変数交互作用 | 2,110 | 2.4e-5 | 3.6e-4 | **0** |外部地域データ（人口・距離・人口比）を含め、**現行モデルが系統的に外している部分集団は一つも存在しない**。`kitchen-sink-combination-search`（生の効果量ベース）とはまったく別の方法論で、同じ結論に到達した。

---# 6. 学習曲線 — 律速しているのは特徴量ではなくデータ量特徴量側が閉じているなら、何が性能を縛っているのか。学習曲線を測る。

In [14]:
rng=np.random.RandomState(0)
curve=[]
for frac in [0.3,0.5,0.7,1.0]:
    lls=[]
    for rep in range(2):
        idx=rng.permutation(len(X))[:int(len(X)*frac)]
        Xs,ys=X.iloc[idx],y[idx]; o=np.zeros(len(idx))
        for tri,vai in StratifiedKFold(5,shuffle=True,random_state=rep).split(Xs,ys):
            m=cb.CatBoostClassifier(**A_PARAMS,iterations=560,random_seed=42,verbose=False,cat_features=obj,task_type="CPU")
            m.fit(Xs.iloc[tri],ys[tri]); o[vai]=m.predict_proba(Xs.iloc[vai])[:,1]
        lls.append(log_loss(ys,o))
    curve.append((int(len(X)*frac), np.mean(lls)))
    print(f"  n={curve[-1][0]:5d} ({frac:.0%})  OOF logloss={curve[-1][1]:.5f}")
sl=(curve[-1][1]-curve[-2][1])/(curve[-1][0]-curve[-2][0])
print(f"\n直近の傾き: {sl:.2e} / 1名  →  全く飽和していない")


  n=  828 (30%)  OOF logloss=0.54384


  n= 1380 (50%)  OOF logloss=0.52320


  n= 1932 (70%)  OOF logloss=0.50927


  n= 2761 (100%)  OOF logloss=0.49578

直近の傾き: -1.63e-05 / 1名  →  全く飽和していない


| n | OOF logloss ||---|---|| 828 (30%) | 0.54384 || 1380 (50%) | 0.52320 || 1932 (70%) | 0.50927 || 2761 (100%) | 0.49578 |**曲線は最後まで直線的に下がり続け、飽和の兆候がまったく無い。**直近の傾きは 1名あたり **-1.6e-5**（`modeling-levers-beat-new-features` が実測した-1.95e-5 と同オーダー）。**このモデルは特徴量律速ではなくデータ量律速**である。20回以上の特徴量探索が失敗し続けた理由は、まさにここにある。

---# 7. ならば Test の 2,502行 を学習に足せるか — 疑似ラベルの実測データ量律速なら、ラベルの無い Test 2,502行（Train比 +91%）を使えれば効くはず。Train 内でその状況を再現して測る：各foldで `V`=評価 / 残りを `L`=ラベルあり(60%) と`U`=ラベルを隠す(40%) に分け、`L のみ` vs `L + 疑似ラベル付きU` を比較する。

In [15]:
def fit_ce(Xa,ya,w=None,seed=42,it=560):
    m=cb.CatBoostClassifier(**A_PARAMS,iterations=it,random_seed=seed,verbose=False,
                            cat_features=obj,task_type="CPU",loss_function="CrossEntropy")
    m.fit(Xa,ya,sample_weight=w); return m

SEEDS3=[42,7,2024]
out={k:[] for k in ["L_only","pseudo_soft_w1.0","pseudo_soft_w0.5","pseudo_soft_w0.25",
                    "pseudo_hard_conf0.8","L+U_true(上限)"]}
rng=np.random.RandomState(0)
for fold,(rest,vai) in enumerate(StratifiedKFold(5,shuffle=True,random_state=0).split(X,y)):
    rest=rng.permutation(rest); k=int(len(rest)*0.6); Li,Ui=rest[:k],rest[k:]
    XL,yL=X.iloc[Li],y[Li]; XU,yU=X.iloc[Ui],y[Ui]; XV,yV=X.iloc[vai],y[vai]
    ev=lambda Xa,ya,w=None: np.mean([fit_ce(Xa,ya,w,s).predict_proba(XV)[:,1] for s in SEEDS3],0)
    out["L_only"].append(log_loss(yV,ev(XL,yL)))
    pu=np.mean([fit_ce(XL,yL,None,s).predict_proba(XU)[:,1] for s in SEEDS3],0)
    Xc=pd.concat([XL,XU])
    for wn,w in [("1.0",1.0),("0.5",0.5),("0.25",0.25)]:
        out[f"pseudo_soft_w{wn}"].append(log_loss(yV, ev(Xc, np.concatenate([yL,pu]),
                                          np.concatenate([np.ones(len(yL)),np.full(len(pu),w)]))))
    conf=(pu>0.8)|(pu<0.2)
    out["pseudo_hard_conf0.8"].append(log_loss(yV, ev(pd.concat([XL,XU[conf]]),
                                       np.concatenate([yL,(pu[conf]>0.5).astype(float)]))))
    out["L+U_true(上限)"].append(log_loss(yV, ev(Xc, np.concatenate([yL,yU]))))
    print(f"  fold{fold}: n_L={len(Li)} n_U={len(Ui)} n_V={len(vai)} 高信頼U={conf.sum()}")

b=np.mean(out["L_only"])
print("\n=== 疑似ラベルの効果（5-fold平均）===")
for k,v in out.items(): print(f"  {k:20s} {np.mean(v):.5f}  (対 L_only {np.mean(v)-b:+.5f})")


  fold0: n_L=1324 n_U=884 n_V=553 高信頼U=413


  fold1: n_L=1325 n_U=884 n_V=552 高信頼U=399


  fold2: n_L=1325 n_U=884 n_V=552 高信頼U=384


  fold3: n_L=1325 n_U=884 n_V=552 高信頼U=352


  fold4: n_L=1325 n_U=884 n_V=552 高信頼U=393

=== 疑似ラベルの効果（5-fold平均）===
  L_only               0.50934  (対 L_only +0.00000)
  pseudo_soft_w1.0     0.51115  (対 L_only +0.00181)
  pseudo_soft_w0.5     0.51014  (対 L_only +0.00080)
  pseudo_soft_w0.25    0.51116  (対 L_only +0.00182)
  pseudo_hard_conf0.8  0.51769  (対 L_only +0.00835)
  L+U_true(上限)         0.49028  (対 L_only -0.01906)


**結果（不採用）**| 構成 | logloss | 対 L_only ||---|---|---|| L_only | 0.50934 | — || pseudo soft w=1.0 | 0.51115 | **+0.0018** || pseudo soft w=0.5 | 0.51014 | **+0.0008** || pseudo soft w=0.25 | 0.51116 | **+0.0018** || pseudo hard conf0.8 | 0.51769 | **+0.0084** || **L+U 真ラベル（上限）** | 0.49028 | **-0.0191** |884行の追加は**真のラベルなら -0.019** の価値があるのに、疑似ラベルは**4変種すべてで悪化側**（+0.0008〜+0.0084）で、その価値を全く回収できない。重みを下げても高信頼だけに絞っても改善しない。**データ量律速だと分かっても、そのデータは作れない。** 半教師あり方向は閉じた。

---# 8. 念のためのリーク点検（先入観を外す＝ここも見る）

In [16]:
tp=train_persona.copy(); tp["idn"]=tp["社員ID"].str[1:].astype(int); tp["row"]=np.arange(len(tp))
tp["depn"]=tp["初期部署ID"].str.rsplit("_",n=1).str[1].astype(int)
print("社員ID × label : r=%+.4f p=%.3g" % stats.pointbiserialr(y,tp.idn))
print("CSV行順  × label : r=%+.4f p=%.3g" % stats.pointbiserialr(y,tp.row))
print("部署連番 × label : r=%+.4f" % np.corrcoef(tp.depn,y)[0,1])
print("ID と 入社日 の相関: %.4f （ID採番は入社順ではない）" % np.corrcoef(tp.idn, pd.to_datetime(tp["入社日"]).astype("int64"))[0,1])
print("train/test の社員ID重複:", len(set(tp.idn) & set(test_persona["社員ID"].str[1:].astype(int))))
for c in ["部署ID","上司ID"]:
    a,b=set(train_monthly[c].dropna()),set(test_monthly[c].dropna())
    print(f"{c}: train {len(a)} / test {len(b)} / 共通 {len(a&b)} ({len(a&b)/len(b):.1%} of test)")


社員ID × label : r=-0.0088 p=0.646
CSV行順  × label : r=-0.0086 p=0.651
部署連番 × label : r=-0.0042
ID と 入社日 の相関: 0.0083 （ID採番は入社順ではない）
train/test の社員ID重複: 0
部署ID: train 465 / test 464 / 共通 462 (99.6% of test)
上司ID: train 1556 / test 1507 / 共通 1313 (87.1% of test)


**結果**: 社員ID・CSV行順・部署連番のいずれもラベルと無相関（|r|<0.01）。採番順のリークは無い。上司IDは Test の 87.1%、部署IDは 99.6% が Train と共有されているが、両者とも v5/v6 で「職種の再パッケージ」「二項ノイズの1.12倍」と決着済み。

---# 9. 結論## 判定サマリー| # | 検証 | 判定 | 根拠 ||---|---|---|---|| 1 | 共変量シフトの全数監査 | **新発見（精緻化）** | 動くのは給与のみ。条件付きシフトは **+0.9〜1.1 sd**（周辺分布の d=0.25 では見えない） || 2 | 給与コホート正規化 | **不採用** | MAD 0.0146〜0.0159 < ノイズ床 0.02122。第95節を独立に追認 || 3 | 外部賃金統計（性別・年齢別） | **不採用（決定的）** | 本データに性別賃金差なし(0.96〜1.02)・性別行動差なし・賃金上昇率が実日本の**10倍** || 4 | 外部地域データ（人口・距離） | **不採用** | 残差スキャンで有意差ゼロ || 5 | テキストのテンプレート構造 | **不採用** | 35,599文中35,591文がユニーク＝LLM生成の自由文。テンプレート集合は存在しない || 6 | **残差スキャン（単変量94・ペア953）** | **決定的な null** | Bonferroni通過 **0件**。未捕捉の部分集団は存在しない || 7 | **学習曲線** | **重要（新発見）** | 全く飽和せず。傾き -1.8e-5/名 ＝ **データ量律速** || 8 | 疑似ラベル（半教師あり） | **不採用** | 4変種すべて悪化(+0.003〜+0.010)。真ラベル上限 -0.018 の1%も回収できず || 9 | ID・行順・部署連番のリーク | **null** | 全て \|r\|<0.01 |## 総括**特徴量側は本当に閉じている。** 従来の「生の効果量スキャン」とは独立な方法論（残差スキャン）で1,047通りを検定して通過ゼロ、という形で確認できた。外部データも、ユーザー提案の性別・年齢別賃金を含めて、**本データの生成器が実世界の賃金構造を使っていない**ため原理的に効かない。一方で **学習曲線が飽和していない**という新しい事実が出た。20回以上の特徴量探索が失敗し続けた機構的な理由はこれである（`monthly-data-information-ceiling` が月次データについて示したことの、モデル全体版）。ただし疑似ラベルでその不足を埋めることはできなかった。**したがって残る打ち手は `private-lb-variance-strategy` の通り、新規性ではなく分散削減（プール平均・モデルクラスの多様化）に限られる**、という既存の方針が本ノートブックによって改めて裏づけられた。